# L7c: Restricted Boltzmann Machines
In this lecture, we'll continue our discussion of Boltzmann Machines by introducing the __Restricted Boltzmann Machine (RBM)__. 

A Restricted Boltzmann Machine (RBM) is a simplified version of the general Boltzmann machine that removes connections between nodes in the same layer. This restriction creates a bipartite graph structure that enables efficient training and sampling algorithms.

> __Learning Objectives__
>
> * __Define a restricted Boltzmann machine:__ Describe the bipartite graph structure of an RBM with visible and hidden layers, and explain how the absence of intra-layer connections simplifies the network.
> * __Implement feedforward and feedback passes:__ Apply the conditional independence property to compute hidden states from visible inputs and reconstruct visible states from hidden activations.
> * __Explain contrastive divergence training:__ Describe the CD-k algorithm for approximating the gradient of the log-likelihood and updating the weights and biases of the RBM.

Let's get started!

___

## Examples
Today, we will use the following examples to illustrate key concepts:

> [▶ Understanding a Small Boltzmann Machines](CHEME-5820-L7a-Example-SmallBoltzmannMachine-Spring-2026.ipynb). In this example, we will explore some questions surrounding the sampling and stationary distribution of a _small_ Boltzmann machine. In particular, we'll look at one of the key limitations of any training approach, namely requiring convergence to a stationary distribution for each training iteration.
___

<div>
    <center>
      <img
        src="figs/Fig-Boltzmann-Machine-Schematic.svg"
        alt="triangle with all three sides equal"
        height="200"
        width="600"
      />
    </center>
  </div>

## Review: Boltzmann Machines
Formally, [a Boltzmann Machine](https://en.wikipedia.org/wiki/Boltzmann_machine) $\mathcal{B}$ is a fully connected _undirected weighted graph_ defined by the tuple $\mathcal{B} = \left(\mathcal{V},\mathcal{E}, \mathbf{W},\mathbf{b}, \mathbf{s}\right)$.
* __Units__: Each unit (vertex, node, neuron) $v_{i}\in\mathcal{V}$ has a binary state (`on` or `off`) and a bias value 
$b_{i}\in\mathbb{R}$. The bias vector $\mathbf{b}\in\mathbb{R}^{|\mathcal{V}|}$ is the vector of bias values for all nodes in the network.
    - A machine $\mathcal{B}$ may have _visible_ nodes (the state is visible) and _hidden_ nodes (the state is not visible to us). The visible nodes represent the state of the data, while the hidden nodes capture the underlying structure of the data (latent variables).
    - The set of all nodes in the machine $\mathcal{B}$ is denoted by $\mathcal{V} \equiv \left\{v_{1},v_{2},\ldots,v_{|\mathcal{V}|}\right\}$, where $|\mathcal{V}|$ is the number of nodes in the network. We can partition the set of nodes into visible nodes $\mathcal{V}_{\text{vis}}$ and hidden nodes $\mathcal{V}_{\text{hid}}$, such that $\mathcal{V} = \mathcal{V}_{\text{vis}} \cup \mathcal{V}_{\text{hid}}$ and $\mathcal{V}_{\text{vis}} \cap \mathcal{V}_{\text{hid}} = \emptyset$.
* __Edges__: Each edge $e\in\mathcal{E}$ has a weight. The weight of the edge connecting $v_{i}\in\mathcal{V}$ and $v_{j}\in\mathcal{V}$, is denoted by $w_{ij}\in\mathbf{W}$, where the weight matrix $\mathbf{W}\in\mathbb{R}^{|\mathcal{V}|\times|\mathcal{V}|}$ is symmetric, i.e. $w_{ij} = w_{ji}$ and $w_{ii} = 0$ (no self loops). The weights $w_{ij}\in\mathbb{R}$ determine the strength of the connection between nodes $i$ and $j$. 
* __States__: The state of the machine $\mathcal{B}$ is represented by a binary vector $\mathbf{s}\in\mathbb{R}^{|\mathcal{V}|}$, where $s_{i}\in\{-1,1\}$ is the state of node $v_{i}$. When $s_{i} = 1$, the node is `on`, and when $s_{i} = -1$, the node is `off`. The set of all possible _state configurations_ is denoted by $\mathcal{S} \equiv \left\{\mathbf{s}^{(1)},\mathbf{s}^{(2)},\ldots,\mathbf{s}^{(N)}\right\}$, where $N$ is the number of possible state configurations, or $N = 2^{|\mathcal{V}|}$ for binary units.

Let's take another look at the Lab L7b from a different perspective. 

> [▶ Understanding a Small Boltzmann Machines](CHEME-5820-L7a-Example-SmallBoltzmannMachine-Spring-2026.ipynb). In this example, we will explore some questions surrounding the sampling and stationary distribution of a _small_ Boltzmann machine. In particular, we'll look at one of the key limitations of any training approach, namely requiring convergence to a stationary distribution for each training iteration.

Now that we have formally defined the machine $\mathcal{B}$, let's take a look at a restricted version of this machine, the Restricted Boltzmann Machine (RBM).

___

<div>
    <center>
      <img
        src="figs/Fig-Restricted-Boltzmann-Machine-Schematic.svg"
        alt="triangle with all three sides equal"
        height="200"
        width="600"
      />
    </center>
</div>

## Restricted Boltzmann Machines (RBMs)
A general Boltzmann machine is a fully connected undirected graph where every node connects to every other node. A Restricted Boltzmann Machine simplifies this structure by organizing nodes into two layers: a visible layer and a hidden layer, with connections only between layers.

Formally, an RBM is defined by the tuple $\mathcal{R} = \left(\mathcal{V}, \mathcal{H}, \mathbf{W}, \mathbf{a}, \mathbf{b}\right)$:

* **Visible layer** $\mathcal{V}$: The set of visible units $\left\{v_{1}, v_{2}, \ldots, v_{n}\right\}$ where $n = |\mathcal{V}|$. Each visible unit has a binary state $v_{i} \in \{-1, 1\}$ and a bias $a_{i} \in \mathbb{R}$. The bias vector is $\mathbf{a} \in \mathbb{R}^{n}$.
* **Hidden layer** $\mathcal{H}$: The set of hidden units $\left\{h_{1}, h_{2}, \ldots, h_{m}\right\}$ where $m = |\mathcal{H}|$. Each hidden unit has a binary state $h_{j} \in \{-1, 1\}$ and a bias $b_{j} \in \mathbb{R}$. The bias vector is $\mathbf{b} \in \mathbb{R}^{m}$.
* **Weight matrix** $\mathbf{W} \in \mathbb{R}^{n \times m}$: The weight $w_{ij}$ represents the connection strength between visible unit $v_{i}$ and hidden unit $h_{j}$.

The restriction is that there are no connections within the visible layer (no $v_{i}$-$v_{j}$ edges) and no connections within the hidden layer (no $h_{i}$-$h_{j}$ edges). This bipartite structure is the source of the name "restricted."

### Objective Function and Energy
Up to this point, we have defined the RBM architecture. The next question is the learning objective: what should $\mathbf{W}$, $\mathbf{a}$, and $\mathbf{b}$ be chosen to do? The modeling goal is to make the RBM assign high probability to the training patterns we actually observe.

Let the dataset be $\mathbf{X}=\left\{\mathbf{x}^{(1)},\mathbf{x}^{(2)},\dots,\mathbf{x}^{(m)}\right\}$ with $\mathbf{x}^{(r)}\in\{-1,1\}^{n}$. A standard objective is to minimize
$$
D_{\mathrm{KL}}\left(\hat p_{\mathrm{data}}\,\|\,p_{\theta}\right).
$$
Here $D_{\mathrm{KL}}$ is the Kullback-Leibler divergence. For discrete visible states, it is
$$
D_{\mathrm{KL}}\left(\hat p_{\mathrm{data}}\,\|\,p_{\theta}\right)
=\sum_{\mathbf{v}}\hat p_{\mathrm{data}}(\mathbf{v})\log\frac{\hat p_{\mathrm{data}}(\mathbf{v})}{p_{\theta}(\mathbf{v})}.
$$
This quantity is always nonnegative and equals zero only when the two distributions match exactly. So, minimizing it means making the model distribution $p_{\theta}$ look as much like the empirical data distribution $\hat p_{\mathrm{data}}$ as possible.

Since the entropy term of $\hat p_{\mathrm{data}}$ does not depend on $\theta=\{\mathbf{W},\mathbf{a},\mathbf{b}\}$, minimizing KL divergence is equivalent to maximum likelihood:
$$
\max_{\theta}\;\mathcal{L}(\theta),\qquad
\mathcal{L}(\theta)=\frac{1}{m}\sum_{r=1}^{m}\log p_{\theta}\!\left(\mathbf{x}^{(r)}\right).
$$

The objective is controlled by the RBM energy function:
$$
E_{\theta}(\mathbf{v},\mathbf{h})=-\mathbf{a}^{\top}\mathbf{v}-\mathbf{b}^{\top}\mathbf{h}-\mathbf{v}^{\top}\mathbf{W}\mathbf{h}
$$
with joint distribution
$$
p_{\theta}(\mathbf{v},\mathbf{h})=\frac{1}{Z_{\theta}}\exp\!\left(-\beta E_{\theta}(\mathbf{v},\mathbf{h})\right),
\qquad
Z_{\theta}=\sum_{\mathbf{v}',\mathbf{h}'}\exp\!\left(-\beta E_{\theta}(\mathbf{v}',\mathbf{h}')\right).
$$
The visible-model probability is obtained by marginalizing hidden states:
$$
p_{\theta}(\mathbf{v})=\sum_{\mathbf{h}}p_{\theta}(\mathbf{v},\mathbf{h})=
\frac{1}{Z_{\theta}}\sum_{\mathbf{h}}\exp\!\left(-\beta E_{\theta}(\mathbf{v},\mathbf{h})\right).
$$

For one training pattern $\mathbf{v}$, the log-likelihood gradient is
$$
\frac{\partial\log p_{\theta}(\mathbf{v})}{\partial\theta}
= -\beta\,\mathbb{E}_{p(\mathbf{h}\mid\mathbf{v})}\!\left[\frac{\partial E_{\theta}(\mathbf{v},\mathbf{h})}{\partial\theta}\right]
+ \beta\,\mathbb{E}_{p(\mathbf{v}',\mathbf{h}')}\!\left[\frac{\partial E_{\theta}(\mathbf{v}',\mathbf{h}')}{\partial\theta}\right].
$$
Using
$$
\frac{\partial E}{\partial w_{ij}}=-v_{i}h_{j},\qquad
\frac{\partial E}{\partial a_{i}}=-v_{i},\qquad
\frac{\partial E}{\partial b_{j}}=-h_{j},
$$
we obtain the familiar difference-of-expectations form:
$$
\frac{\partial\log p_{\theta}(\mathbf{v})}{\partial w_{ij}}
=\beta\left(\langle v_{i}h_{j}\rangle_{\text{data}}-\langle v_{i}h_{j}\rangle_{\text{model}}\right),
$$
$$
\frac{\partial\log p_{\theta}(\mathbf{v})}{\partial a_{i}}
=\beta\left(\langle v_{i}\rangle_{\text{data}}-\langle v_{i}\rangle_{\text{model}}\right),
\qquad
\frac{\partial\log p_{\theta}(\mathbf{v})}{\partial b_{j}}
=\beta\left(\langle h_{j}\rangle_{\text{data}}-\langle h_{j}\rangle_{\text{model}}\right).
$$

This derivation gives the training story in one line: increase parameters to match data-driven correlations and decrease parameters that are only supported by the model. The remaining challenge is computational: the model expectations are expensive because they require sampling from $p_{\theta}(\mathbf{v},\mathbf{h})$. That is exactly why we introduce block Gibbs sampling and CD-$k$ next.


### Sampling and CD-$k$ for RBMs (Adapted from General Boltzmann Machines)
In the general Boltzmann machine, exact training is expensive because Gibbs updates are node-wise and the model-dependent statistics ideally require near-stationary samples at each parameter update. An RBM keeps the same likelihood objective, but the bipartite structure removes within-layer couplings. That single structural change gives us efficient block Gibbs sampling and a practical CD-$k$ training rule.

__RBM Sampling (Block Gibbs)__

__Initialize__: choose parameters $\mathbf{W},\mathbf{a},\mathbf{b}$, an initial visible state $\mathbf{v}^{(0)}$, inverse temperature $\beta$, and the number of sampling turns $T$.

For each turn $t=1,2,\dots,T$:

1. Sample all hidden units in parallel from
$$
P\left(h_{j}^{(t)}=1\mid\mathbf{v}^{(t-1)}\right)=\left(1+\exp\left(-2\beta\left((\mathbf{W}^{\top}\mathbf{v}^{(t-1)})_{j}+b_{j}\right)\right)\right)^{-1}
$$
2. Sample all visible units in parallel from
$$
P\left(v_{i}^{(t)}=1\mid\mathbf{h}^{(t)}\right)=\left(1+\exp\left(-2\beta\left((\mathbf{W}\mathbf{h}^{(t)})_{i}+a_{i}\right)\right)\right)^{-1}
$$
3. Store $\left(\mathbf{v}^{(t)},\mathbf{h}^{(t)}\right)$ and continue.

__Output__: the sample chain $\left\{\mathbf{v}^{(t)},\mathbf{h}^{(t)}\right\}_{t=1}^{T}$.

The same block Gibbs mechanism is used inside contrastive divergence.

__RBM Contrastive Divergence (CD-$k$)__

__Initialize__: set $\mathbf{W},\mathbf{a},\mathbf{b}$ to small random values; choose learning rate $\eta$, inverse temperature $\beta=1$, Gibbs depth $k$, tolerance $\epsilon$, and maximum epochs $N_{\text{epochs}}$.

For each epoch $e=1,2,\dots,N_{\text{epochs}}$:

1. For each training pattern $\mathbf{x}^{(r)}\in\mathbf{X}$, set $\mathbf{v}^{(0)}=\mathbf{x}^{(r)}$.
2. Positive phase (data statistics): compute $\mathbf{p}^{+}=P(\mathbf{h}=1\mid\mathbf{v}^{(0)})$, sample $\mathbf{h}^{(0)}\sim P(\mathbf{h}\mid\mathbf{v}^{(0)})$, and form $\langle v_i h_j\rangle^{+}=v_i^{(0)}p_j^{+}$.
3. Negative phase (model statistics): run $k$ block Gibbs steps
$$
\mathbf{v}^{(t)}\sim P(\mathbf{v}\mid\mathbf{h}^{(t-1)}),\qquad
\mathbf{h}^{(t)}\sim P(\mathbf{h}\mid\mathbf{v}^{(t)}),\qquad t=1,\dots,k
$$
then compute $\mathbf{p}^{-}=P(\mathbf{h}=1\mid\mathbf{v}^{(k)})$ and $\langle v_i h_j\rangle^{-}=v_i^{(k)}p_j^{-}$.
4. Update parameters:
$$
\mathbf{W}\leftarrow\mathbf{W}+\eta\left(\langle\mathbf{v}\mathbf{h}^{\top}\rangle^{+}-\langle\mathbf{v}\mathbf{h}^{\top}\rangle^{-}\right)
$$
$$
\mathbf{a}\leftarrow\mathbf{a}+\eta\left(\langle\mathbf{v}\rangle_{\text{data}}-\langle\mathbf{v}\rangle_{\text{model}}\right),\qquad
\mathbf{b}\leftarrow\mathbf{b}+\eta\left(\langle\mathbf{h}\rangle_{\text{data}}-\langle\mathbf{h}\rangle_{\text{model}}\right)
$$
5. Evaluate stopping criteria; if satisfied, terminate, otherwise continue.

__Stopping Criteria__

1. Parameter-change threshold:
$$
\|\Delta\mathbf{W}\|_{F}<\epsilon,\qquad
\|\Delta\mathbf{a}\|_{2}<\epsilon,\qquad
\|\Delta\mathbf{b}\|_{2}<\epsilon
$$
2. Expectation matching:
$$
\max_{i,j}\left|\langle v_{i}h_{j}\rangle^{+}-\langle v_{i}h_{j}\rangle^{-}\right|<\epsilon
$$
$$
\max_{i}\left|\langle v_i\rangle_{\text{data}}-\langle v_i\rangle_{\text{model}}\right|<\epsilon,\qquad
\max_{j}\left|\langle h_j\rangle_{\text{data}}-\langle h_j\rangle_{\text{model}}\right|<\epsilon
$$
3. Fixed budget: stop after $N_{\text{epochs}}$ epochs.

CD-$k$ still trades exactness for speed, but in RBMs each Gibbs step is cheap because both conditional updates are layer-wise and parallelizable.

___


## Lab
In the lab this week, we will implement RBM training with contrastive divergence and examine how the Gibbs depth $k$ changes reconstruction quality and learning stability. [Let's work through the CD training workflow in detail](CHEME-5820-L7a-Training-BoltzmannMachines-Spring-2026.ipynb).


## Summary
This module introduced Boltzmann machines as stochastic recurrent neural networks that define probability distributions over binary state configurations.

> __Key Takeaways:__
>
> * **Boltzmann machine structure:** A Boltzmann machine is a fully connected undirected graph where each node has a binary state and bias, and edges have symmetric weights. The network defines a joint probability distribution over all possible state configurations.
> * **Gibbs sampling for state generation:** Because the partition function is intractable to compute directly, we use Gibbs sampling to generate states from the network. Each node updates stochastically based on the states of its neighbors and the logistic function of its total input.
> * **Convergence to stationary distribution:** After sufficient iterations, the sampled states converge to the Boltzmann distribution. Convergence can be monitored by tracking energy values across sliding windows.

The contrastive divergence algorithm enables training of Boltzmann machines by approximating likelihood gradients without computing the partition function.

___